# ROCStories GPT Pipeline

Single notebook combining all pipeline stages:
- **Section 1** — Configuration
- **Section 2** — Model definition (`model.py`)
- **Section 3** — Data preparation (`prepare.py`) — downloads from Hugging Face, skip if `.bin` files already exist
- **Section 4** — Training setup & helpers
- **Section 5** — Training loop (`train.py`)
- **Section 6** — Standalone evaluation (`eval.py`)

Run top-to-bottom on first use. On resume, skip Section 3.

In [ ]:
import os
import time
import math
import pickle
import re
import shutil
import inspect
from contextlib import nullcontext
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
from datasets import load_dataset

## 1. Configuration

In [ ]:
# I/O
out_dir = 'out-rocstories'
init_from = 'scratch'  # 'scratch' or 'resume'
always_save_checkpoint = True
eval_only = False

# wandb
wandb_log = False
wandb_project = 'rocstories'
wandb_run_name = 'rocstories'

# data
dataset = 'rocstories'
data_dir = os.path.join('data', dataset)

# model architecture
block_size = 256
n_layer = 6
n_head = 6
n_embd = 384
bias = False

# training
batch_size = 32
gradient_accumulation_steps = 4  # effective batch = 32 * 4 = 128
dtype = 'bfloat16'
device = 'cuda'
compile = False

# regularization
dropout = 0.2
weight_decay = 0.2

# optimizer
learning_rate = 1e-4
min_lr = 1e-5
beta1 = 0.9
beta2 = 0.95
grad_clip = 1.0

# lr schedule
decay_lr = True
warmup_iters = 400
lr_decay_iters = 4000

# iterations
max_iters = 4000
eval_interval = 400
eval_iters = 400
log_interval = 50

# system setup
os.makedirs(out_dir, exist_ok=True)
torch.manual_seed(1337)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
print(f'device: {device} | dtype: {dtype}')

## 2. Model Definition

In [ ]:
class LayerNorm(nn.Module):
    """ LayerNorm with optional bias. """
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)


class LayerRMSNorm(nn.Module):
    def __init__(self, ndim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))

    def forward(self, input):
        return F.rms_norm(input, self.weight.shape, self.weight, 1e-5)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print('WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0')
            self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                                         .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        if self.flash:
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0,
                is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerRMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerRMSNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True


class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        print('number of parameters: %.2fM' % (self.get_num_params() / 1e6,))

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f'Cannot forward sequence of length {t}, block size is only {self.config.block_size}'
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def crop_block_size(self, block_size):
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:, :, :block_size, :block_size]

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        print(f'decayed params: {sum(p.numel() for p in decay_params):,}')
        print(f'non-decayed params: {sum(p.numel() for p in nodecay_params):,}')
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas,
                                       **(dict(fused=True) if use_fused else {}))
        print(f'using fused AdamW: {use_fused}')
        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd // cfg.n_head, cfg.block_size
        flops_per_iter = (6 * N + 12 * L * H * Q * T) * T * fwdbwd_per_iter
        mfu = flops_per_iter * (1.0 / dt) / 312e12  # A100 bfloat16 peak
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('Model classes defined.')

## 3. Data Preparation

Downloads ROCStories from Hugging Face and saves `train.bin`, `val.bin`, `test.txt`, `meta.pkl`  
to `data/rocstories/`. **Skip this cell if the files already exist.**

In [ ]:
def story_text(example: dict) -> str:
    if 'story' in example and example['story']:
        return str(example['story']).strip()
    sentence_keys = [
        key for key in ('sentence1', 'sentence2', 'sentence3', 'sentence4', 'sentence5')
        if key in example and example[key]
    ]
    if sentence_keys:
        return ' '.join(str(example[key]).strip() for key in sentence_keys).strip()
    if 'text' in example and example['text']:
        return str(example['text']).strip()
    return ''


def tokenize_split(split, encoder) -> np.ndarray:
    ids = []
    for example in split:
        text = story_text(example)
        if not text:
            continue
        ids.extend(encoder.encode_ordinary(text))
        ids.append(encoder.eot_token)
    return np.array(ids, dtype=np.uint16)


print('Downloading ROCStories from Hugging Face ...')
hf_dataset = load_dataset('mintujupally/ROCStories')

test_split  = hf_dataset['test']
train_val   = hf_dataset['train'].train_test_split(test_size=0.10, seed=1337, shuffle=True)
train_split = train_val['train']
val_split   = train_val['test']

print(f'  train : {len(train_split):,} stories')
print(f'  val   : {len(val_split):,} stories')
print(f'  test  : {len(test_split):,} stories')

enc      = tiktoken.get_encoding('gpt2')
base_dir = os.path.join('data', 'rocstories')
os.makedirs(base_dir, exist_ok=True)

for name, split in {'train': train_split, 'val': val_split}.items():
    print(f'Tokenising {name} split ...')
    ids = tokenize_split(split, enc)
    out_path = os.path.join(base_dir, f'{name}.bin')
    ids.tofile(out_path)
    print(f'  -> {out_path}  ({len(ids):,} tokens, {ids.nbytes / 1e6:.1f} MB)')

meta_path = os.path.join(base_dir, 'meta.pkl')
with open(meta_path, 'wb') as f:
    pickle.dump({'vocab_size': enc.n_vocab, 'tokenizer': 'gpt2'}, f)
print(f'Meta saved -> {meta_path}')

test_path = os.path.join(base_dir, 'test.txt')
with open(test_path, 'w', encoding='utf-8') as f:
    for example in test_split:
        text = story_text(example)
        if text:
            f.write(text + '\n\n')
print(f'Test text saved -> {test_path}')
print('Done.')

## 4. Training Setup

In [ ]:
# ── Story-aligned data loader ──────────────────────────────────────────────
EOT_TOKEN = 50256

def compute_story_starts(data):
    eot_positions = np.where(np.array(data, dtype=np.int32) == EOT_TOKEN)[0]
    starts = np.concatenate([[0], eot_positions + 1])
    return starts[starts + block_size < len(data)]

print('Pre-computing story start positions ...')
_train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
_val_data   = np.memmap(os.path.join(data_dir, 'val.bin'),   dtype=np.uint16, mode='r')
story_starts = {
    'train': compute_story_starts(_train_data),
    'val':   compute_story_starts(_val_data),
}
print(f'  train: {len(story_starts["train"]):,} story starts')
print(f'  val  : {len(story_starts["val"]):,} story starts')
del _train_data, _val_data


def get_batch(split):
    data   = np.memmap(os.path.join(data_dir, f'{split}.bin'), dtype=np.uint16, mode='r')
    starts = story_starts[split]
    chosen = np.random.randint(0, len(starts), size=batch_size)
    ix     = starts[chosen]
    x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64))         for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


# ── Loss estimator ─────────────────────────────────────────────────────────
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# ── Inline PPL evaluator (replaces subprocess eval.py call) ───────────────
@torch.no_grad()
def evaluate_ppl(input_file=None):
    if input_file is None:
        input_file = os.path.join(data_dir, 'test.txt')
    if not os.path.exists(input_file):
        print(f'evaluate_ppl: {input_file} not found, skipping.')
        return None
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()
    paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
    enc_eval = tiktoken.get_encoding('gpt2')
    encode   = lambda s: enc_eval.encode(s, allowed_special={'<|endoftext|>'})
    total_nll, total_tokens = 0.0, 0
    model.eval()
    with ctx:
        for para in paragraphs:
            token_ids = encode(para)
            if len(token_ids) < 2:
                continue
            pos = 0
            para_pred_tokens = len(token_ids) - 1
            while pos < para_pred_tokens:
                inp = token_ids[pos: pos + block_size]
                tgt = token_ids[pos + 1: pos + 1 + block_size]
                if not tgt:
                    break
                if len(inp) != len(tgt):
                    inp = inp[:len(tgt)]
                x = torch.tensor(inp, dtype=torch.long, device=device)[None, :]
                y = torch.tensor(tgt, dtype=torch.long, device=device)[None, :]
                _, loss = model(x, y)
                n_tok = len(tgt)
                total_nll    += loss.item() * n_tok
                total_tokens += n_tok
                pos += n_tok
    model.train()
    if total_tokens == 0:
        return None
    return math.exp(total_nll / total_tokens)


# ── LR schedule ────────────────────────────────────────────────────────────
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


# ── Model init ─────────────────────────────────────────────────────────────
iter_num       = 0
best_val_loss  = 1e9
best_eval_ppl  = float('inf')

meta_path = os.path.join(data_dir, 'meta.pkl')
meta_vocab_size = None
if os.path.exists(meta_path):
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    meta_vocab_size = meta['vocab_size']
    print(f'vocab_size = {meta_vocab_size}')

model_args = dict(n_layer=n_layer, n_head=n_head, n_embd=n_embd,
                  block_size=block_size, bias=bias, vocab_size=None, dropout=dropout)

if init_from == 'scratch':
    print('Initializing model from scratch')
    model_args['vocab_size'] = meta_vocab_size if meta_vocab_size is not None else 50304
    model = GPT(GPTConfig(**model_args))

elif init_from == 'resume':
    print(f'Resuming from {out_dir}')
    ckpt_path  = os.path.join(out_dir, 'ckpt.pt')
    checkpoint = torch.load(ckpt_path, map_location=device)
    for k in ['n_layer', 'n_head', 'n_embd', 'block_size', 'bias', 'vocab_size']:
        model_args[k] = checkpoint['model_args'][k]
    model      = GPT(GPTConfig(**model_args))
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    iter_num      = checkpoint['iter_num']
    best_val_loss = checkpoint['best_val_loss']
    best_eval_ppl = checkpoint.get('best_eval_ppl', float('inf'))

if block_size < model.config.block_size:
    model.crop_block_size(block_size)
    model_args['block_size'] = block_size

model.to(device)
scaler    = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device_type)
if init_from == 'resume':
    optimizer.load_state_dict(checkpoint['optimizer'])

if wandb_log:
    import wandb
    wandb.init(project=wandb_project, name=wandb_run_name)

print('Setup complete.')

## 5. Training Loop

Interrupt the kernel (`■` button) to stop early. Progress is checkpointed every `eval_interval` steps.

In [ ]:
X, Y             = get_batch('train')
t0               = time.time()
local_iter_num   = 0
running_mfu      = -1.0
got_best_val_loss = False

try:
    while True:
        # set lr
        lr = get_lr(iter_num) if decay_lr else learning_rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # eval + checkpoint
        if iter_num % eval_interval == 0:
            losses = estimate_loss()
            print(
                f'step {iter_num}: '
                f'train loss {losses["train"]:.4f}, '
                f'val loss {losses["val"]:.4f}, '
                f'ppl {math.exp(losses["val"]):.2f}'
            )
            if wandb_log:
                import wandb
                wandb.log({'iter': iter_num, 'train/loss': losses['train'],
                           'val/loss': losses['val'], 'lr': lr, 'mfu': running_mfu * 100})

            got_best_val_loss = losses['val'] < 3.3 and losses['val'] < best_val_loss
            if got_best_val_loss:
                best_val_loss = losses['val']
                if iter_num > 0:
                    ckpt = {
                        'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'model_args': model_args,
                        'iter_num': iter_num,
                        'best_val_loss': best_val_loss,
                        'best_eval_ppl': best_eval_ppl,
                    }
                    torch.save(ckpt, os.path.join(out_dir, 'ckpt.pt'))
                    print(f'  checkpoint saved -> {out_dir}/ckpt.pt')
            else:
                print(f'  val loss {losses["val"]:.4f} not improved (best: {best_val_loss:.4f}), skipping checkpoint')

        # external PPL eval (inline, no subprocess)
        if iter_num % eval_interval == 0 and iter_num > 0 and got_best_val_loss:
            ppl = evaluate_ppl()
            if ppl is not None:
                print(f'  eval ppl: {ppl:.4f}')
                if ppl < best_eval_ppl:
                    best_eval_ppl = ppl
                    src = os.path.join(out_dir, 'ckpt.pt')
                    dst = os.path.join(out_dir, 'ckpt_best.pt')
                    if os.path.exists(src):
                        shutil.copyfile(src, dst)
                        print(f'  new best ppl {best_eval_ppl:.4f} -> {dst}')

        if iter_num == 0 and eval_only:
            break

        # forward / backward
        for micro_step in range(gradient_accumulation_steps):
            with ctx:
                _, loss = model(X, Y)
                loss = loss / gradient_accumulation_steps
            X, Y = get_batch('train')
            scaler.scale(loss).backward()

        if grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        t1 = time.time()
        dt = t1 - t0
        t0 = t1

        if iter_num % log_interval == 0:
            lossf = loss.item() * gradient_accumulation_steps
            if local_iter_num >= 5:
                mfu = model.estimate_mfu(batch_size * gradient_accumulation_steps, dt)
                running_mfu = mfu if running_mfu == -1.0 else 0.9 * running_mfu + 0.1 * mfu
            print(f'iter {iter_num}: loss {lossf:.4f}, time {dt * 1000:.2f}ms, mfu {running_mfu * 100:.2f}%')

        iter_num      += 1
        local_iter_num += 1

        if iter_num > max_iters:
            print('Training complete.')
            break

except KeyboardInterrupt:
    print('Training interrupted by user.')

## 6. Standalone Evaluation

Loads `ckpt_best.pt` (falls back to `ckpt.pt`) and reports PPL on `test.txt`.  
Can be run independently after training.

In [ ]:
# load best checkpoint
ckpt_path = os.path.join(out_dir, 'ckpt_best.pt')
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
print(f'Loading checkpoint: {ckpt_path}')

checkpoint  = torch.load(ckpt_path, map_location=device)
eval_model  = GPT(GPTConfig(**checkpoint['model_args']))
state_dict  = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
eval_model.load_state_dict(state_dict)
eval_model.eval()
eval_model.to(device)

# evaluate on test.txt
input_file = os.path.join('data', 'rocstories', 'test.txt')
with open(input_file, 'r', encoding='utf-8') as f:
    content = f.read()
paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
print(f'Loaded {len(paragraphs)} test stories from {input_file}')

enc_eval = tiktoken.get_encoding('gpt2')
encode   = lambda s: enc_eval.encode(s, allowed_special={'<|endoftext|>'})

total_nll, total_tokens, skipped = 0.0, 0, 0
eval_block_size = eval_model.config.block_size

with torch.no_grad():
    with ctx:
        for para in paragraphs:
            token_ids = encode(para)
            if len(token_ids) < 2:
                skipped += 1
                continue
            pos = 0
            para_pred_tokens = len(token_ids) - 1
            while pos < para_pred_tokens:
                inp = token_ids[pos: pos + eval_block_size]
                tgt = token_ids[pos + 1: pos + 1 + eval_block_size]
                if not tgt:
                    break
                if len(inp) != len(tgt):
                    inp = inp[:len(tgt)]
                x = torch.tensor(inp, dtype=torch.long, device=device)[None, :]
                y = torch.tensor(tgt, dtype=torch.long, device=device)[None, :]
                _, loss = eval_model(x, y)
                n_tok = len(tgt)
                total_nll    += loss.item() * n_tok
                total_tokens += n_tok
                pos += n_tok

avg_loss = total_nll / total_tokens
ppl      = math.exp(avg_loss)

print('-' * 35)
print(f'checkpoint      : {ckpt_path}')
print(f'stories used    : {len(paragraphs) - skipped}')
print(f'stories skipped : {skipped}')
print(f'pred tokens     : {total_tokens:,}')
print(f'avg loss        : {avg_loss:.4f}')
print(f'ppl             : {ppl:.2f}')